# What's Cooking? — Cuisine Classification

## tl;dr

This notebook predicts a recipe's cuisine from its ingredient list. It compares a majority-class baseline, Multinomial Naive Bayes, Logistic Regression, and a linear SVM using grouped, stratified cross-validation. It then refits the validation-selected model on all labeled recipes and writes a Kaggle-format submission.

In the executed three-fold grouped validation, **Logistic Regression** was selected with **76.01% accuracy** and **0.6808 macro F1**. Multinomial Naive Bayes reached 74.58% accuracy, Linear SVM reached 74.01%, and the majority baseline reached 19.71%.

Run this notebook from top to bottom. It reads only the raw files in `data/` and writes derived charts, tables, and predictions to `outputs/`.

## Context & Methods

### Problem definition

The Kaggle *What's Cooking?* task is a supervised **multiclass classification** problem. Each training recipe has a cuisine label and a list of ingredients; each test recipe has only its ingredients. The goal is to predict one of 20 cuisines for every test recipe.

### Key assumptions

- A recipe's ingredient phrases are the predictive input; `id` is an identifier, not a feature.
- Ingredient order is not treated as meaningful. Phrases are lowercased, trimmed, de-duplicated within a recipe, and kept intact (for example, `soy sauce` remains one feature).
- `StratifiedGroupKFold` groups identical normalized ingredient sets so duplicates do not appear in both a validation fold and its training fold.
- The vocabulary and every model are fitted only on the training portion of each fold, avoiding validation/test leakage.

Primary model-selection metric: validation accuracy (aligned with the competition). Macro F1 and per-class scores are reported because cuisines are imbalanced.

In [16]:
# 1. Imports, paths, and reproducibility
from pathlib import Path
from collections import Counter
import json

import matplotlib
matplotlib.use('Agg')  # enables reproducible plot saving in non-interactive environments
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             ConfusionMatrixDisplay, confusion_matrix, f1_score)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
# Three grouped folds are a practical runtime compromise for this course notebook.
N_SPLITS = 3

current_dir  = Path.cwd().resolve()
if current_dir.name == 'notebooks':
    part_dir = current_dir.parent
elif current_dir.name == 'part-1-whats-cooking':
    part_dir = current_dir
elif (current_dir / 'part-1-whats-cooking').exists():
    part_dir = current_dir / 'part-1-whats-cooking'
else:
    raise FileNotFoundError(
        f"Could not locate part-1-whats-cooking from working directory: {current_dir}"
    )

project_dir = part_dir.parent
data_dir = part_dir / 'data'
output_dir = part_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / 'train.json'
test_path = data_dir / 'test.json'

assert train_path.exists(), f'Train file not found: {train_path}'
assert test_path.exists(), f'Test file not found: {test_path}'

print('Project structure located successfully.')
print('Training data: part-1-whats-cooking/data/train.json')
print('Test data: part-1-whats-cooking/data/test.json')
print('Outputs: part-1-whats-cooking/outputs/')

Project structure located successfully.
Training data: part-1-whats-cooking/data/train.json
Test data: part-1-whats-cooking/data/test.json
Outputs: part-1-whats-cooking/outputs/


## Data

### Load the competition files

The raw JSON files are not changed. Each recipe is loaded into a pandas DataFrame only for inspection and modeling.

In [4]:
# 2. Load data
with train_path.open(encoding='utf-8') as file:
    train = pd.DataFrame(json.load(file))
with test_path.open(encoding='utf-8') as file:
    test = pd.DataFrame(json.load(file))

print('Training shape:', train.shape)
print('Test shape:', test.shape)
print('Training columns:', train.columns.tolist())
print('Test columns:', test.columns.tolist())
print('\nExample training recipes:')
print(train[['id', 'cuisine', 'ingredients']].head(3).to_string(index=False))

Training shape: (39774, 3)
Test shape: (9944, 2)
Training columns: ['id', 'cuisine', 'ingredients']
Test columns: ['id', 'ingredients']

Example training recipes:
   id     cuisine                                                                                                                                          ingredients
10259       greek                       [romaine lettuce, black olives, grape tomatoes, garlic, pepper, purple onion, seasoning, garbanzo beans, feta cheese crumbles]
25693 southern_us                [plain flour, ground pepper, salt, tomatoes, ground black pepper, thyme, eggs, green tomatoes, yellow corn meal, milk, vegetable oil]
20130    filipino [eggs, pepper, salt, mayonaise, cooking oil, green chilies, grilled chicken breasts, garlic powder, yellow onion, soy sauce, butter, chicken livers]


### Dataset overview and quality checks

These checks establish the recipe-level grain, class balance, missingness, and duplicate ingredient-list patterns before modeling.

In [5]:
# 3. Overview, missing values, duplicates, and ingredient counts
train_missing = train.isna().sum()
test_missing = test.isna().sum()
ordered_duplicate_rows = train.duplicated(subset=['ingredients']).sum()

def normalize_ingredients(ingredients):
    """Keep ingredient phrases intact while standardizing trivial text variation."""
    return sorted({ingredient.strip().lower() for ingredient in ingredients if ingredient and ingredient.strip()})

train['normalized_ingredients'] = train['ingredients'].apply(normalize_ingredients)
test['normalized_ingredients'] = test['ingredients'].apply(normalize_ingredients)
train['recipe_group'] = train['normalized_ingredients'].apply(tuple)
test['recipe_group'] = test['normalized_ingredients'].apply(tuple)
set_duplicate_rows = train.duplicated(subset=['recipe_group']).sum()

train['ingredient_count'] = train['ingredients'].str.len()
test['ingredient_count'] = test['ingredients'].str.len()

print('Number of cuisine classes:', train['cuisine'].nunique())
print('\nMissing values in training data:\n', train_missing.to_string())
print('\nMissing values in test data:\n', test_missing.to_string())
print(f'\nExtra rows with duplicate ordered ingredient lists: {ordered_duplicate_rows}')
print(f'Extra rows with duplicate normalized ingredient sets: {set_duplicate_rows}')
print('\nIngredient-count summary:')
print(pd.DataFrame({'train': train['ingredient_count'].describe(),
                    'test': test['ingredient_count'].describe()}).round(2).to_string())

Number of cuisine classes: 20

Missing values in training data:
 id             0
cuisine        0
ingredients    0

Missing values in test data:
 id             0
ingredients    0

Extra rows with duplicate ordered ingredient lists: 100
Extra rows with duplicate normalized ingredient sets: 532

Ingredient-count summary:
          train     test
count  39774.00  9944.00
mean      10.77    10.80
std        4.43     4.47
min        1.00     1.00
25%        8.00     8.00
50%       10.00    10.00
75%       13.00    13.00
max       65.00    50.00


## Exploratory Data Analysis

The following four charts are saved into `outputs/` as PNG files. The cuisine/ingredient heatmap emphasizes ingredients that occur disproportionately often within a cuisine rather than universally common items such as salt.

In [6]:
# 4. Cuisine class distribution
cuisine_counts = train['cuisine'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(cuisine_counts.index, cuisine_counts.values, color='#4C78A8')
ax.set_title('Training Recipes by Cuisine')
ax.set_xlabel('Number of recipes')
ax.set_ylabel('Cuisine')
fig.tight_layout()
fig.savefig(output_dir / 'cuisine_class_distribution.png', dpi=150)
plt.close(fig)
print(cuisine_counts.sort_values(ascending=False).to_string())

italian         7838
mexican         6438
southern_us     4320
indian          3003
chinese         2673
french          2646
cajun_creole    1546
thai            1539
japanese        1423
greek           1175
spanish          989
korean           830
vietnamese       825
moroccan         821
british          804
filipino         755
irish            667
jamaican         526
russian          489
brazilian        467


In [7]:
# 5. Ingredient-count distribution
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.arange(0.5, max(train['ingredient_count'].max(), test['ingredient_count'].max()) + 1.5, 1)
ax.hist(train['ingredient_count'], bins=bins, alpha=0.70, label='Train', color='#4C78A8')
ax.hist(test['ingredient_count'], bins=bins, alpha=0.55, label='Test', color='#F58518')
ax.axvline(train['ingredient_count'].median(), color='#1F3B5D', linestyle='--', label='Train median')
ax.set_xlim(0, 35)
ax.set_title('Ingredient Counts per Recipe')
ax.set_xlabel('Ingredients in recipe (x-axis capped at 35 for readability)')
ax.set_ylabel('Number of recipes')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'ingredient_count_distribution.png', dpi=150)
plt.close(fig)

In [8]:
# 6. Most common ingredient phrases
all_ingredients = [ingredient for recipe in pd.concat([train['normalized_ingredients'], test['normalized_ingredients']])
                   for ingredient in recipe]
common_ingredients = pd.Series(Counter(all_ingredients)).sort_values(ascending=False).head(20).sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(common_ingredients.index, common_ingredients.values, color='#54A24B')
ax.set_title('20 Most Common Ingredient Phrases')
ax.set_xlabel('Number of recipes containing ingredient')
fig.tight_layout()
fig.savefig(output_dir / 'most_common_ingredients.png', dpi=150)
plt.close(fig)
print(common_ingredients.sort_values(ascending=False).to_string())

salt                   22533
onions                 10008
olive oil               9888
water                   9293
garlic                  9171
sugar                   8064
garlic cloves           7771
butter                  6077
ground black pepper     5989
all-purpose flour       5816
vegetable oil           5516
pepper                  5508
eggs                    4262
soy sauce               4120
kosher salt             3930
green onions            3817
tomatoes                3812
large eggs              3700
carrots                 3542
unsalted butter         3474


In [9]:
# 7. Signature ingredients by cuisine (prevalence lift)
exploded = train[['cuisine', 'normalized_ingredients']].explode('normalized_ingredients')
exploded = exploded.rename(columns={'normalized_ingredients': 'ingredient'})
by_cuisine = pd.crosstab(exploded['cuisine'], exploded['ingredient'])
within_cuisine_rate = by_cuisine.div(cuisine_counts.reindex(by_cuisine.index), axis=0)
overall_rate = by_cuisine.sum(axis=0) / len(train)
lift = within_cuisine_rate.div(overall_rate + 1e-12, axis=1)
# Avoid highlighting ingredients supported by only a handful of recipes.
lift = lift.where(by_cuisine >= 10)
signature_ingredients = lift.idxmax(axis=1)
signature_table = pd.DataFrame({
    'cuisine': signature_ingredients.index,
    'ingredient': signature_ingredients.values,
    'within_cuisine_rate': [within_cuisine_rate.loc[cuisine, ingredient] for cuisine, ingredient in signature_ingredients.items()],
    'lift_overall': [lift.loc[cuisine, ingredient] for cuisine, ingredient in signature_ingredients.items()]
}).sort_values('cuisine')
signature_table.to_csv(output_dir / 'signature_ingredients_by_cuisine.csv', index=False)

heatmap_values = within_cuisine_rate.loc[signature_table['cuisine'], signature_table['ingredient']].to_numpy()
fig, ax = plt.subplots(figsize=(12, 8))
image = ax.imshow(heatmap_values, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(len(signature_table)))
ax.set_xticklabels(signature_table['ingredient'], rotation=90)
ax.set_yticks(range(len(signature_table)))
ax.set_yticklabels(signature_table['cuisine'])
ax.set_title('Cuisine-Specific Signature Ingredient Prevalence')
fig.colorbar(image, ax=ax, label='Share of cuisine recipes containing ingredient')
fig.tight_layout()
fig.savefig(output_dir / 'cuisine_signature_ingredients.png', dpi=150)
plt.close(fig)
print(signature_table.round(3).to_string(index=False))

     cuisine            ingredient  within_cuisine_rate  lift_overall
   brazilian               cachaca                0.150        85.169
     british        beef drippings                0.015        49.470
cajun_creole           file powder                0.030        24.152
     chinese fermented black beans                0.013        14.880
    filipino       calamansi juice                0.023        52.681
      french              armagnac                0.005        13.958
       greek                  ouzo                0.009        33.850
      indian             fenugreek                0.012        13.245
       irish         irish whiskey                0.045        57.708
     italian               gnocchi                0.005         5.075
    jamaican  jamaican jerk season                0.040        72.179
    japanese          dashi powder                0.011        27.951
      korean             gochugaru                0.036        47.920
     mexican        

## Feature Representation

Each recipe becomes a sparse binary multi-hot vector: one column per normalized ingredient phrase, with value 1 when that phrase appears in the recipe. This retains meaningful phrases such as `fish sauce` and `olive oil`, while avoiding a large, less interpretable word-token vocabulary.

The ID is excluded because it does not describe a recipe and could produce accidental patterns. The vectorizer is inside each pipeline, so it learns its vocabulary from each training fold only.

In [10]:
# 8. Feature helper and grouped validation setup
def identity(document):
    return document

def make_vectorizer():
    return CountVectorizer(analyzer=identity, lowercase=False, binary=True, dtype=np.int8)

X = train['normalized_ingredients'].tolist()
y = train['cuisine'].to_numpy()
groups = train['recipe_group'].astype(str).to_numpy()
X_test = test['normalized_ingredients'].tolist()

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
print(f'Using {N_SPLITS}-fold StratifiedGroupKFold with {len(np.unique(groups)):,} ingredient-set groups.')
print('No vectorizer is fit on validation or test recipes during model comparison.')

Using 3-fold StratifiedGroupKFold with 39,242 ingredient-set groups.
No vectorizer is fit on validation or test recipes during model comparison.


## Modeling and Evaluation

The four models below are deliberately simple, strong baselines for sparse text-like features. All use the same folds. The linear models use balanced class weights as a modest response to the long-tailed class distribution; accuracy remains the selection criterion.

In [11]:
# 9. Compare models with out-of-fold predictions
models = {
    'Majority-class baseline': Pipeline([
        ('vectorizer', make_vectorizer()),
        ('classifier', DummyClassifier(strategy='most_frequent'))
    ]),
    'Multinomial Naive Bayes': Pipeline([
        ('vectorizer', make_vectorizer()),
        ('classifier', MultinomialNB(alpha=0.5))
    ]),
    'Logistic Regression': Pipeline([
        ('vectorizer', make_vectorizer()),
        # Native multinomial Logistic Regression for the 20-class task.
        ('classifier', LogisticRegression(C=1.0, solver='lbfgs', max_iter=500,
                                          class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Linear SVM': Pipeline([
        ('vectorizer', make_vectorizer()),
        ('classifier', LinearSVC(C=1.0, class_weight='balanced', random_state=RANDOM_STATE,
                                 dual='auto'))
    ])
}

results = []
oof_predictions = {}
for name, model in models.items():
    print(f'Evaluating {name}...')
    predictions = cross_val_predict(model, X, y, cv=cv, groups=groups, method='predict')
    oof_predictions[name] = predictions
    results.append({
        'model': name,
        'accuracy': accuracy_score(y, predictions),
        'macro_f1': f1_score(y, predictions, average='macro')
    })

results_df = pd.DataFrame(results).sort_values(['accuracy', 'macro_f1'], ascending=False).reset_index(drop=True)
results_df.to_csv(output_dir / 'model_comparison.csv', index=False)
print(results_df.round(4).to_string(index=False))

best_model_name = results_df.loc[0, 'model']
best_oof_predictions = oof_predictions[best_model_name]
print(f'\nSelected by validation accuracy: {best_model_name}')

Evaluating Majority-class baseline...
Evaluating Multinomial Naive Bayes...
Evaluating Logistic Regression...
Evaluating Linear SVM...
                  model  accuracy  macro_f1
    Logistic Regression    0.7601    0.6808
Multinomial Naive Bayes    0.7458    0.6539
             Linear SVM    0.7401    0.6501
Majority-class baseline    0.1971    0.0165

Selected by validation accuracy: Logistic Regression


In [12]:
# 10. Best-model class-level evaluation and normalized confusion matrix
class_report = pd.DataFrame(classification_report(y, best_oof_predictions, output_dict=True, zero_division=0)).T
class_report.index.name = 'cuisine'
class_report.to_csv(output_dir / 'best_model_per_class_report.csv')
print(class_report.loc[sorted(train['cuisine'].unique())].round(3).to_string())

labels = sorted(train['cuisine'].unique())
normalized_cm = confusion_matrix(y, best_oof_predictions, labels=labels, normalize='true')
fig, ax = plt.subplots(figsize=(12, 10))
display_cm = ConfusionMatrixDisplay(confusion_matrix=normalized_cm, display_labels=labels)
display_cm.plot(ax=ax, cmap='Blues', xticks_rotation=90, colorbar=True, values_format='.2f')
ax.set_title(f'Normalized Confusion Matrix: {best_model_name}')
fig.tight_layout()
fig.savefig(output_dir / 'best_model_normalized_confusion_matrix.png', dpi=150)
plt.close(fig)

confusion_without_diagonal = normalized_cm.copy()
np.fill_diagonal(confusion_without_diagonal, 0)
confusion_pairs = []
for actual_index, predicted_index in zip(*np.where(confusion_without_diagonal > 0)):
    confusion_pairs.append({
        'actual_cuisine': labels[actual_index],
        'predicted_cuisine': labels[predicted_index],
        'row_normalized_rate': confusion_without_diagonal[actual_index, predicted_index]
    })
confusion_pairs = pd.DataFrame(confusion_pairs).sort_values('row_normalized_rate', ascending=False)
confusion_pairs.to_csv(output_dir / 'most_common_confusions.csv', index=False)
print('\nMost common confusions:')
print(confusion_pairs.head(10).round(3).to_string(index=False))

              precision  recall  f1-score  support
cuisine                                           
brazilian         0.540   0.638     0.585    467.0
british           0.411   0.499     0.451    804.0
cajun_creole      0.725   0.735     0.730   1546.0
chinese           0.828   0.808     0.818   2673.0
filipino          0.573   0.695     0.628    755.0
french            0.585   0.621     0.603   2646.0
greek             0.667   0.740     0.702   1175.0
indian            0.881   0.871     0.876   3003.0
irish             0.439   0.559     0.492    667.0
italian           0.868   0.805     0.835   7838.0
jamaican          0.667   0.732     0.698    526.0
japanese          0.787   0.696     0.739   1423.0
korean            0.761   0.789     0.775    830.0
mexican           0.930   0.884     0.906   6438.0
moroccan          0.710   0.783     0.745    821.0
russian           0.359   0.546     0.433    489.0
southern_us       0.761   0.721     0.741   4320.0
spanish           0.463   0.546

In [13]:
# 11. Interpret influential ingredients when the selected model is linear
best_model = models[best_model_name]
best_model.fit(X, y)
classifier = best_model.named_steps['classifier']
feature_names = best_model.named_steps['vectorizer'].get_feature_names_out()

if hasattr(classifier, 'coef_'):
    coefficient_matrix = np.asarray(classifier.coef_)
    influential_rows = []
    for cuisine, coefficients in zip(classifier.classes_, coefficient_matrix):
        top_indices = np.argsort(coefficients)[-10:][::-1]
        for rank, index in enumerate(top_indices, start=1):
            influential_rows.append({
                'cuisine': cuisine,
                'rank': rank,
                'ingredient': feature_names[index],
                'model_weight': coefficients[index]
            })
    influential_ingredients = pd.DataFrame(influential_rows)
    influential_ingredients.to_csv(output_dir / 'best_model_influential_ingredients.csv', index=False)
    print(influential_ingredients.groupby('cuisine').head(3).round(3).to_string(index=False))
else:
    print(f'{best_model_name} does not expose linear feature coefficients; no coefficient table was saved.')

     cuisine  rank                ingredient  model_weight
   brazilian     1                   cachaca         7.011
   brazilian     2              manioc flour         5.168
   brazilian     3             tapioca flour         4.359
     british     1                   stilton         4.282
     british     2            stilton cheese         4.222
     british     3            beef drippings         3.490
cajun_creole     1           cajun seasoning         4.439
cajun_creole     2          creole seasoning         3.920
cajun_creole     3            creole mustard         2.363
     chinese     1          mandarin oranges         3.467
     chinese     2 chinese five-spice powder         3.384
     chinese     3      szechwan peppercorns         3.287
    filipino     1           calamansi juice         4.146
    filipino     2              lumpia skins         3.215
    filipino     3           lumpia wrappers         3.159
      french     1            gruyere cheese         2.7

## Final Model and Test Predictions

The model with the highest grouped-validation accuracy is selected. It is refit on all 39,774 labeled recipes, then used to predict the 9,944 test recipes. This final refit is appropriate because no test labels are used and model selection was completed first.

In [14]:
# 12. Refit selected model on all training data and create submission
final_model = models[best_model_name]
final_model.fit(X, y)
test_predictions = final_model.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'cuisine': test_predictions})
assert len(submission) == len(test)
assert submission['id'].is_unique
assert set(submission['cuisine']).issubset(set(train['cuisine']))
submission.to_csv(output_dir / 'submission.csv', index=False)

print('Saved predictions to', output_dir / 'submission.csv')
print(submission.head().to_string(index=False))

Saved predictions to C:\Liliya\Courses\SJSU\Data Mining\cmpe255-assignment-1\part-1-whats-cooking\outputs\submission.csv
   id      cuisine
18009      british
28583  southern_us
41580      italian
29752 cajun_creole
35687      italian


## Results and Interpretation

Run the following cell after the preceding evaluation cells. It summarizes the observed validation results, easiest and hardest cuisines by F1, and the most frequent confusion directions.

### Limitations

- Cuisine is not determined only by ingredients; recipes with similar ingredient sets can legitimately have different labels.
- Ingredient strings retain branded names, spelling variants, and near-synonyms. The deliberately light normalization does not resolve all of these.
- Grouped cross-validation is more conservative than a random split, but it cannot guarantee that the held-out Kaggle test distribution is identical to training.
- Linear phrase features do not model quantities, preparation methods, or deeper interactions between ingredients.

In [15]:
# 13. Compact results narrative for presentation
per_class_f1 = class_report.loc[labels, 'f1-score'].sort_values()
print('Validation model comparison:')
print(results_df.round(4).to_string(index=False))
print(f'\nBest model: {best_model_name}')
best_accuracy = results_df.loc[0, 'accuracy']
best_macro_f1 = results_df.loc[0, 'macro_f1']
print(f'Best validation accuracy: {best_accuracy:.4f}')
print(f'Best validation macro F1: {best_macro_f1:.4f}')
print('\nEasiest cuisines by F1:')
print(per_class_f1.tail(5).round(3).to_string())
print('\nHardest cuisines by F1:')
print(per_class_f1.head(5).round(3).to_string())
print('\nTop confusion directions:')
print(confusion_pairs.head(5).round(3).to_string(index=False))
print('\nDerived artifacts are available in:', output_dir)

Validation model comparison:
                  model  accuracy  macro_f1
    Logistic Regression    0.7601    0.6808
Multinomial Naive Bayes    0.7458    0.6539
             Linear SVM    0.7401    0.6501
Majority-class baseline    0.1971    0.0165

Best model: Logistic Regression
Best validation accuracy: 0.7601
Best validation macro F1: 0.6808

Easiest cuisines by F1:
cuisine
korean     0.775
chinese    0.818
italian    0.835
indian     0.876
mexican    0.906

Hardest cuisines by F1:
cuisine
russian      0.433
british      0.451
irish        0.492
spanish      0.501
brazilian    0.585

Top confusion directions:
actual_cuisine predicted_cuisine  row_normalized_rate
    vietnamese              thai                0.185
        french           italian                0.119
         irish           british                0.118
       british            french                0.113
       british             irish                0.112

Derived artifacts are available in: C:\Liliya\Courses\

## Takeaways

The selected Logistic Regression model achieved 76.01% grouped-validation accuracy, ahead of the other three compared models. Mexican, Indian, Italian, Chinese, and Korean were easiest by F1; Russian, British, Irish, Spanish, and Brazilian were hardest. The largest observed confusion direction was Vietnamese predicted as Thai (18.55% of Vietnamese validation recipes).

This experiment uses an understandable sparse-feature pipeline suitable for a graduate data-mining assignment. The saved validation tables, confusion matrix, and ingredient-coefficient table provide evidence for explaining the final model in a video or report. Future work could compare phrase features against character n-grams or carefully engineered ingredient normalization, but those extensions are intentionally outside this core experiment.